In [2]:
import numpy as np
import pandas as pd
import joblib
import time
import os
import psutil

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [3]:
X_train = np.load("../dataset/processed/X_train.npy")
X_test = np.load("../dataset/processed/X_test.npy")

y_train = np.load("../dataset/processed/y_train.npy")
y_test = np.load("../dataset/processed/y_test.npy")

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (65865, 10)
Testing : (16467, 10)


In [4]:
rf = RandomForestClassifier(random_state=42)

ada = AdaBoostClassifier(random_state=42)

gb = GradientBoostingClassifier(random_state=42)

knn = KNeighborsClassifier()

nb = GaussianNB()

In [5]:
hard_voting = VotingClassifier(

    estimators=[

        ('rf', rf),

        ('ada', ada),

        ('gb', gb),

        ('knn', knn),

        ('nb', nb)

    ],

    voting='hard'

)

soft_voting = VotingClassifier(

    estimators=[

        ('rf', rf),

        ('ada', ada),

        ('gb', gb),

        ('knn', knn),

        ('nb', nb)

    ],

    voting='soft'

)

stacking = StackingClassifier(

    estimators=[

        ('rf', rf),

        ('ada', ada),

        ('gb', gb),

        ('knn', knn),

        ('nb', nb)

    ],

    final_estimator=LogisticRegression(),

    cv=5

)

combined_ensemble = StackingClassifier(

    estimators=[

        ('rf', rf),

        ('ada', ada),

        ('gb', gb),

        ('knn', knn),

        ('nb', nb)

    ],

    final_estimator=GradientBoostingClassifier(),

    cv=5

)

In [6]:
ensembles = {

    "Hard Voting": hard_voting,

    "Soft Voting": soft_voting,

    "Stacking": stacking,

    "Combined Ensemble": combined_ensemble

}

In [7]:
results = []

In [14]:
import os
import time
import joblib
import psutil

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

results = []

for name, model in ensembles.items():

    print("=" * 60)
    print(f"Training {name}...")

    # Process information
    process = psutil.Process(os.getpid())

    # -----------------------------
    # Training
    # -----------------------------
    cpu_before = psutil.cpu_percent(interval=0.1)

    start_train = time.perf_counter()

    model.fit(X_train, y_train)

    end_train = time.perf_counter()

    cpu_after = psutil.cpu_percent(interval=0.1)

    # Current RAM usage (MB)
    ram_usage = process.memory_info().rss / (1024 * 1024)

    # -----------------------------
    # Prediction
    # -----------------------------
    start_predict = time.perf_counter()

    predictions = model.predict(X_test)

    end_predict = time.perf_counter()

    # Average prediction time per sample (milliseconds)
    prediction_time_ms = (
        (end_predict - start_predict)
        / len(X_test)
    ) * 1000

    # -----------------------------
    # Evaluation Metrics
    # -----------------------------
    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions
    )

    recall = recall_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()

    specificity = tn / (tn + fp)

    fpr = fp / (fp + tn)

    roc = roc_auc_score(
        y_test,
        predictions
    )

    # -----------------------------
    # Save Model
    # -----------------------------
    filename = "../models/" + \
        name.replace(" ", "_") + ".pkl"

    joblib.dump(model, filename)

    model_size = os.path.getsize(filename) / (1024 * 1024)

    # -----------------------------
    # Store Results
    # -----------------------------
    results.append({

        "Model": name,

        "Accuracy": round(accuracy, 4),

        "Precision": round(precision, 4),

        "Recall": round(recall, 4),

        "F1": round(f1, 4),

        "Specificity": round(specificity, 4),

        "FPR": round(fpr, 4),

        "ROC_AUC": round(roc, 4),

        "CPU_Usage (%)": round(cpu_after, 2),

        "RAM_Usage_MB": round(ram_usage, 2),

        "Training_Time (s)": round(
            end_train - start_train,
            4
        ),

        "Prediction_Time (ms/sample)": round(
            prediction_time_ms,
            4
        ),

        "Model_Size_MB": round(
            model_size,
            2
        )

    })

    print(f"✓ {name} completed.")

print("=" * 60)
print("All ensemble models evaluated successfully!")

Training Hard Voting...
✓ Hard Voting completed.
Training Soft Voting...
✓ Soft Voting completed.
Training Stacking...
✓ Stacking completed.
Training Combined Ensemble...
✓ Combined Ensemble completed.
All ensemble models evaluated successfully!


In [15]:
ensemble_results = pd.DataFrame(results)

ensemble_results

,Model,Accuracy,Precision,Recall,F1,Specificity,FPR,ROC_AUC,CPU_Usage (%),RAM_Usage_MB,Training_Time (s),Prediction_Time (ms/sample),Model_Size_MB
0,Hard Voting,0.9150,0.9771,0.8659,0.9181,0.9751,0.0249,0.9205,27.1,110.12,18.1443,0.0832,64.52
1,Soft Voting,0.9195,0.9796,0.8721,0.9227,0.9777,0.0223,0.9249,13.2,175.36,15.1955,0.0825,64.52
2,Stacking,0.9351,0.9531,0.9279,0.9403,0.9441,0.0559,0.9360,22.9,260.46,73.5512,0.0677,64.52
3,Combined Ensemble,0.9350,0.9627,0.9175,0.9396,0.9565,0.0435,0.9370,22.4,325.48,88.7267,0.0675,64.65


In [16]:
ensemble_results = ensemble_results.sort_values(

    by="Accuracy",

    ascending=False

)

ensemble_results

,Model,Accuracy,Precision,Recall,F1,Specificity,FPR,ROC_AUC,CPU_Usage (%),RAM_Usage_MB,Training_Time (s),Prediction_Time (ms/sample),Model_Size_MB
2,Stacking,0.9351,0.9531,0.9279,0.9403,0.9441,0.0559,0.9360,22.9,260.46,73.5512,0.0677,64.52
3,Combined Ensemble,0.9350,0.9627,0.9175,0.9396,0.9565,0.0435,0.9370,22.4,325.48,88.7267,0.0675,64.65
1,Soft Voting,0.9195,0.9796,0.8721,0.9227,0.9777,0.0223,0.9249,13.2,175.36,15.1955,0.0825,64.52
0,Hard Voting,0.9150,0.9771,0.8659,0.9181,0.9751,0.0249,0.9205,27.1,110.12,18.1443,0.0832,64.52


In [17]:
ensemble_results.to_csv(

    "../results/ensemble_metrics.csv",

    index=False

)

print("CSV Saved")

CSV Saved


In [18]:
ensemble_results.to_excel(

    "../results/ensemble_metrics.xlsx",

    index=False

)

print("Excel Saved")

Excel Saved


In [19]:
winner = ensemble_results.iloc[0]

print("="*60)

print("BEST ENSEMBLE")

print("="*60)

print(winner)

BEST ENSEMBLE
Model                          Stacking
Accuracy                         0.9351
Precision                        0.9531
Recall                           0.9279
F1                               0.9403
Specificity                      0.9441
FPR                              0.0559
ROC_AUC                           0.936
CPU_Usage (%)                      22.9
RAM_Usage_MB                     260.46
Training_Time (s)               73.5512
Prediction_Time (ms/sample)      0.0677
Model_Size_MB                     64.52
Name: 2, dtype: object
